# Geographic task - model analysis

Per-target-species inspection (non_target dropped). SincNet: which frequencies matter, via weight magnitudes folded back to the sinc channels. mel CNN: which spectrogram regions matter, via Grad-CAM.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
from building.polychirp_training.training import TinyChirpModel

COLLECTION = "task_s3_49_2_r50"
NON_TARGET = "non_target"
SINCNET = TinyChirpModel(base_model="sincnet", multi=1.0)
SINCNET_NEW = TinyChirpModel(base_model="sincnet_new", multi=1.0)
SINC_MODELS = [SINCNET, SINCNET_NEW]  # raw-waveform sinc models
MEL = TinyChirpModel(base_model="mel_cnn_new", multi=1.0)
GRADCAM_CLIPS = 24  # clips per species for the mel heatmap

## Classes & model loading

Output logit `j` is `species[j]` (targets only). `load` picks each model's own `SincnetConv`.

In [ ]:
import importlib

import numpy as np
import keras

from building.polychirp_training.training import load_dataset_catalog, build_dataset_from_catalog

catalog = load_dataset_catalog(COLLECTION)
non_target_idx = catalog.class_names.index(NON_TARGET)
target_idxs = [i for i, n in enumerate(catalog.class_names) if n != NON_TARGET]
species = [catalog.class_names[i] for i in target_idxs]


def load(model):
    path = model.paths(COLLECTION).model_path
    if not path.exists():
        print(f"missing {path}")
        return None
    mod = importlib.import_module(f"building.models.{model.name}")
    custom = {"SincnetConv": mod.SincnetConv} if hasattr(mod, "SincnetConv") else {}
    return keras.models.load_model(path, custom_objects=custom, compile=False)


print(f"{len(species)} target species: {species}")

## SincNet: recursive frequency importance

`channel_importance` folds each Dense/Conv2D weight back onto the sinc channels, giving one value per filter.

In [ ]:
import matplotlib.pyplot as plt
from keras import layers as L


def channel_importance(model, target_idx):
    """Importance of each sinc channel for output target_idx, folded back through
    every Dense/Conv2D layer by squared weight magnitude. A Flatten is un-flattened
    and its spatial axes summed, so heads with or without a final global pool both
    reduce to one value per sinc filter."""
    imp = np.zeros(model.output_shape[-1])
    imp[target_idx] = 1.0
    for layer in reversed(model.layers):
        if isinstance(layer, L.Dense):
            imp = (layer.get_weights()[0] ** 2) @ imp
        elif isinstance(layer, L.Flatten):
            shape = tuple(layer.input.shape[1:])
            imp = imp.reshape(shape).sum(axis=tuple(range(len(shape) - 1)))
        elif isinstance(layer, L.Conv2D):
            imp = (layer.get_weights()[0] ** 2).sum(axis=(0, 1)) @ imp
        if layer.name == "sincnet_convolution":
            break
    return imp


def sinc_importance(model):
    """(f_low, f_high, sr, {species: importance}) for a loaded sinc model, each
    species' vector normalised to its own max, or None if missing."""
    m = load(model)
    if m is None:
        return None
    sinc = m.get_layer("sincnet_convolution")
    sr = sinc.sample_rate
    f_low = np.abs(sinc.f1.numpy()) * sr
    f_high = f_low + np.abs(sinc.band.numpy()) * sr
    imp = {s: channel_importance(m, j) for j, s in enumerate(species)}
    imp = {s: v / v.max() for s, v in imp.items()}
    return f_low, f_high, sr, imp


sinc_runs = [(m.name, r) for m in SINC_MODELS if (r := sinc_importance(m)) is not None]

### Which channels carry weight

One figure per sinc model, one row per species: each filter as a rectangle over its pass-band, height = importance.

In [ ]:
for name, (f_low, f_high, sr, imp) in sinc_runs:
    nyquist = sr / 2
    fig, axes = plt.subplots(
        len(species), 1, figsize=(11, 2.4 * len(species)), sharex=True, squeeze=False
    )
    for ax, s in zip(axes[:, 0], species):
        v = imp[s]
        for lo, hi, p in zip(f_low, f_high, v):
            ax.add_patch(
                plt.Rectangle(
                    (lo, 0), hi - lo, p,
                    facecolor="steelblue", edgecolor="black", lw=0.4, alpha=0.45,
                )
            )
        ax.axhline(float(np.median(v)), color="red", lw=1, ls="--", label="median")
        ax.set_xlim(0, nyquist)
        ax.set_ylim(0, 1.08)
        ax.set_ylabel(f"{s}\nimportance")
        ax.legend(loc="upper right", fontsize=8, frameon=False)
    axes[-1, 0].set_xlabel("frequency (Hz)")
    fig.suptitle(f"{name}: per-channel frequency importance (full network)  ({COLLECTION})")
    plt.tight_layout()
    plt.show()

### Cumulative importance per frequency

At each frequency, sum the importance of every filter whose pass-band covers it.

In [ ]:
for name, (f_low, f_high, sr, imp) in sinc_runs:
    nyquist = sr / 2
    grid = np.linspace(0, nyquist, 2000)
    covers = (grid[:, None] >= f_low[None, :]) & (grid[:, None] < f_high[None, :])  # (grid, ch)
    fig, ax = plt.subplots(figsize=(11, 4))
    for s in species:
        ax.plot(grid, covers @ imp[s], lw=1.6, label=s)
    ax.set_xlim(0, nyquist)
    ax.set_xlabel("frequency (Hz)")
    ax.set_ylabel("cumulative importance")
    ax.legend(frameon=False)
    ax.set_title(f"{name}: cumulative frequency importance (summed channel rectangles)  ({COLLECTION})")
    plt.tight_layout()
    plt.show()

## mel CNN: Grad-CAM over the spectrogram

Per-species heatmap from the gradient of each logit back to the last conv layer, upsampled and averaged over clips.

In [ ]:
import tensorflow as tf

mel_model = load(MEL)

ds, _ = build_dataset_from_catalog(
    catalog, target_idxs, non_target_idx, 32, np.random.default_rng(0),
    split="test", input_repr=MEL.input_frontend, augment=False,
)

clips = {j: [] for j in range(len(species))}
for batch in ds:
    x, y = batch[0].numpy(), batch[1].numpy()
    for j in range(len(species)):
        need = GRADCAM_CLIPS - len(clips[j])
        if need > 0:
            clips[j].extend(x[y[:, j] == 1][:need])
    if all(len(c) >= GRADCAM_CLIPS for c in clips.values()):
        break
clips = {j: np.stack(c) for j, c in clips.items()}
print({species[j]: len(c) for j, c in clips.items()})

In [ ]:
last_conv = [l for l in mel_model.layers if isinstance(l, L.Conv2D)][-1]
grad_model = keras.Model(mel_model.input, [last_conv.output, mel_model.output])


def gradcam(x, target_idx):
    """Mean Grad-CAM over a batch of clips, returning a (frames, mel) importance map."""
    x = tf.convert_to_tensor(x)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(x)
        loss = preds[:, target_idx]
    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.einsum("nhwc,nc->nhw", conv_out, weights))
    cam = tf.image.resize(cam[..., None], (x.shape[1], x.shape[2]))[..., 0]
    return cam.numpy().mean(0)


cams = {species[j]: gradcam(clips[j], j) for j in range(len(species))}

### Per-species spectrogram heatmaps

Each map normalised to its own peak; frequency axis at mel-bin centres.

In [ ]:
from building.utils import hz_to_mel, mel_to_hz
from building.utils.constants import (
    LOWER_EDGE_HERTZ, UPPER_EDGE_HERTZ, NUM_MEL_BINS_MEL, FRAME_STEP, SAMPLE_RATE,
)

mel_centers = mel_to_hz(
    np.linspace(hz_to_mel(LOWER_EDGE_HERTZ), hz_to_mel(UPPER_EDGE_HERTZ), NUM_MEL_BINS_MEL)
)

fig, axes = plt.subplots(1, len(species), figsize=(5 * len(species), 4.2), squeeze=False)
for ax, s in zip(axes[0], species):
    cam = cams[s] / cams[s].max()
    frames, mels = cam.shape
    duration = frames * FRAME_STEP / SAMPLE_RATE
    im = ax.imshow(
        cam.T, origin="lower", aspect="auto", cmap="magma",
        extent=[0, duration, 0, mels],
    )
    yt = np.linspace(0, mels - 1, 6).astype(int)
    ax.set_yticks(yt + 0.5)
    ax.set_yticklabels([f"{mel_centers[i]:.0f}" for i in yt])
    ax.set_title(s)
    ax.set_xlabel("time (s)")
axes[0, 0].set_ylabel("frequency (Hz)")
fig.colorbar(im, ax=list(axes[0]), label="relative importance")
fig.suptitle(f"mel CNN: Grad-CAM importance over the spectrogram  ({COLLECTION})")
plt.show()